In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Module, Conv2d, Parameter, Softmax
import torchvision.ops as ops
import math
import cv2
import ast


import glob
import numpy as np

import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import torchvision
import torchvision.models as models
import torchvision.datasets as datasets

In [2]:
class Resnet_FPN_split(nn.Module):
    def __init__(self,out_channels = 128):
        super(Resnet_FPN_split, self).__init__()
        self.resnet = models.resnet18(pretrained=False)
        self.resnet.to(device)
        self.layer_1 = nn.Sequential(*list(self.resnet.children())[:4])
        self.layer_2 = self.resnet.layer2
        self.layer_3 = self.resnet.layer3
        self.layer_4 = self.resnet.layer4

        self.lateral4 = nn.Conv2d(512, out_channels, kernel_size=1, stride=1, padding=0)
        self.lateral3 = nn.Conv2d(256, out_channels, kernel_size=1, stride=1, padding=0)
        self.lateral2 = nn.Conv2d(128, out_channels, kernel_size=1, stride=1, padding=0)
        self.lateral1 = nn.Conv2d(64, out_channels, kernel_size=1, stride=1, padding=0)

        self.output4 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.output3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.output2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.output1 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        if (torch.isnan(x).any()):
            print("error at Resnet_FPN")
        c1 = self.layer_1(x)
        if (torch.any(torch.isnan(c1))):
            print('error resnet ')
        c2 = self.layer_2(c1)
        c3 = self.layer_3(c2)
        c4 = self.layer_4(c3)


        return c1 , c2 , c3 , c4

In [4]:
class PAM_Module(nn.Module):
    """ Position attention module"""
    #Ref from SAGAN
    def __init__(self, in_dim):
        super(PAM_Module, self).__init__()
        self.chanel_in = in_dim

        self.query_conv = Conv2d(in_channels=in_dim, out_channels=in_dim//8, kernel_size=1)
        self.key_conv = Conv2d(in_channels=in_dim, out_channels=in_dim//8, kernel_size=1)
        self.value_conv = Conv2d(in_channels=in_dim, out_channels=in_dim, kernel_size=1)
        self.gamma = Parameter(torch.zeros(1))

        self.softmax = Softmax(dim=-1)
    def forward(self, x):
        """
            inputs :
                x : input feature maps( B X C X H X W)
            returns :
                out : attention value + input feature
                attention: B X (HxW) X (HxW)
        """
        m_batchsize, C, height, width = x.size()
        proj_query = self.query_conv(x).view(m_batchsize, -1, width*height).permute(0, 2, 1)
        proj_key = self.key_conv(x).view(m_batchsize, -1, width*height)
        energy = torch.bmm(proj_query, proj_key)
        attention = self.softmax(energy)
        proj_value = self.value_conv(x).view(m_batchsize, -1, width*height)

        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(m_batchsize, C, height, width)

        out = self.gamma*out + x
        return out

In [5]:
class CAM_Module(nn.Module):
    """ Channel attention module"""
    def __init__(self, in_dim):
        super(CAM_Module, self).__init__()
        self.chanel_in = in_dim


        self.gamma = Parameter(torch.zeros(1))
        self.softmax  = Softmax(dim=-1)
    def forward(self,x):
        """
            inputs :
                x : input feature maps( B X C X H X W)
            returns :
                out : attention value + input feature
                attention: B X C X C
        """
        m_batchsize, C, height, width = x.size()
        proj_query = x.view(m_batchsize, C, -1)
        proj_key = x.view(m_batchsize, C, -1).permute(0, 2, 1)
        energy = torch.bmm(proj_query, proj_key)
        energy_new = torch.max(energy, -1, keepdim=True)[0].expand_as(energy)-energy
        attention = self.softmax(energy_new)
        proj_value = x.view(m_batchsize, C, -1)

        out = torch.bmm(attention, proj_value)
        out = out.view(m_batchsize, C, height, width)

        out = self.gamma*out + x
        return out

In [12]:
class CBR(nn.Module):
    '''
    This class defines the convolution layer with batch normalization and PReLU activation
    '''
    def __init__(self, nIn, nOut, kSize, stride=1):
        '''

        :param nIn: number of input channels
        :param nOut: number of output channels
        :param kSize: kernel size
        :param stride: stride rate for down-sampling. Default is 1
        '''
        super().__init__()
        padding = int((kSize - 1)/2)
        #self.conv = nn.Conv2d(nIn, nOut, kSize, stride=stride, padding=padding, bias=False)
        self.conv = nn.Conv2d(nIn, nOut, (kSize, kSize), stride=stride, padding=(padding, padding), bias=False)
        #self.conv1 = nn.Conv2d(nOut, nOut, (1, kSize), stride=1, padding=(0, padding), bias=False)
        self.bn = nn.BatchNorm2d(nOut, eps=1e-03)
        self.act = nn.PReLU(nOut)

    def forward(self, input):
        '''
        :param input: input feature map
        :return: transformed feature map
        '''
        output = self.conv(input)
        #output = self.conv1(output)
        output = self.bn(output)
        output = self.act(output)
        return output
    def fuseforward(self, input):
        output = self.conv(input)
        output = self.act(output)
        return output

In [31]:
class UPx2(nn.Module):
    '''
    This class defines the convolution layer with batch normalization and PReLU activation
    '''
    def __init__(self, nIn, nOut):
        '''

        :param nIn: number of input channels
        :param nOut: number of output channels
        :param kSize: kernel size
        :param stride: stride rate for down-sampling. Default is 1
        '''
        super().__init__()
        self.deconv = nn.ConvTranspose2d(nIn, nOut, 2, stride=2, padding=0, output_padding=0, bias=False)
        self.bn = nn.BatchNorm2d(nOut, eps=1e-03)
        self.act = nn.PReLU(nOut)

    def forward(self, input):
        '''
        :param input: input feature map
        :return: transformed feature map
        '''
        output = self.deconv(input)
        output = self.bn(output)
        output = self.act(output)
        return output
    def fuseforward(self, input):
        output = self.deconv(input)
        output = self.act(output)
        return output

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [9]:
H_SHAPE = 768   
W_SHAPE = 512

backbone = Resnet_FPN_split()   

sa = PAM_Module(32)
sc = CAM_Module(32)

In [27]:
x = torch.randn(1,3,H_SHAPE,W_SHAPE).to(device)
c1, c2 , c3 , c4 = backbone(x)    
print(c1.shape)

torch.Size([1, 64, 192, 128])


In [28]:
sa = PAM_Module(32)
sc = CAM_Module(32)
conv_sa = CBR(32,32,3)
conv_sc = CBR(32,32,3)
classifier = CBR(32, 32, 1, 1)

In [29]:
b3 = CBR(64,32,3)
y = b3(c1)
out_sa=sa(y)
out_sa=conv_sa(out_sa)
out_sc=sc(y)
out_sc=conv_sc(out_sc)
out_s=out_sa+out_sc
classifier = classifier(out_s)
print(out_sa.shape)
print(out_sc.shape)
print(classifier.shape)

torch.Size([1, 32, 192, 128])
torch.Size([1, 32, 192, 128])
torch.Size([1, 32, 192, 128])


In [39]:
class split_model(nn.Module):
    def __init__(self):
        super(split_model, self).__init__()
        self.backbone = Resnet_FPN_split()
        self.b3 = CBR(64,32,3)

        #row_function
        self.sa_row = PAM_Module(32)
        self.conv_sa_row = CBR(32,32,3)
        self.sc_row = CAM_Module(32)
        self.conv_sc_row = CBR(32,32,3)
        self.classifier_row = CBR(32, 32, 1, 1)

        self.up_1_1_row = UPx2(32,16)
        self.up_2_1_row = UPx2(16,8)

        self.up_1_2_row = UPx2(32,16)
        self.up_2_2_row = UPx2(16,8)

        self.classifier_final_row_seg = UPx2(8,2)
        #col_function
        self.sa_col = PAM_Module(32)
        self.conv_sa_col = CBR(32,32,3)
        self.sc_col = CAM_Module(32)
        self.conv_sc_col = CBR(32,32,3)
        self.classifier_col = CBR(32, 32, 1, 1)

        self.up_1_1_col = UPx2(32,16)
        self.up_2_1_col = UPx2(16,8)

        self.up_1_2_col = UPx2(32,16)
        self.up_2_2_col = UPx2(16,8)

        self.classifier_final_col_seg = UPx2(8,2)
    def forward(self, x):
        c1, c2 , c3 , c4 = self.backbone(x)    
        y = self.b3(c1)

        # row_branch
        out_sa_row=self.sa_row(y)
        out_sa_row=self.conv_sa_row(out_sa_row)
        out_sc_row=self.sc_row(y)
        out_sc_row=self.conv_sc_row(out_sc_row)
        out_s_row=out_sa_row+out_sc_row
        classifier_row = self.classifier_row(out_s_row)

        x_row = self.up_1_1_row(classifier_row)
        # x_row = self.up_2_1_row(x_row)
        final_row = self.classifier_final_row_seg(x_row)
        # col_branch
        out_sa_col=self.sa_col(y)
        out_sa_col=self.conv_sa_col(out_sa_col)
        out_sc_col=self.sc_col(y)
        out_sc_col=self.conv_sc_col(out_sc_col)
        out_s_col=out_sa_col+out_sc_col
        classifier_col = self.classifier_col(out_s_col)

        x_col = self.up_1_1_col(classifier_col)
        # x_col = self.up_2_1_col(x_col)
        final_col = self.classifier_final_col_seg(x_col)
        return final_row, final_col


In [40]:
model_test = split_model()  
model_test.to(device)

# x = torch.randn(1,3,H_SHAPE,W_SHAPE).to(device)
# final_row, final_col = model_test(x)

c:\Users\PC\miniconda3\envs\v1\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\PC\miniconda3\envs\v1\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


split_model(
  (backbone): Resnet_FPN_split(
    (resnet): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn

In [41]:
x = torch.randn(1,3,H_SHAPE,W_SHAPE).to(device)
final_row, final_col = model_test(x)

RuntimeError: Given transposed=1, weight of size [8, 2, 2, 2], expected input[1, 16, 384, 256] to have 8 channels, but got 16 channels instead

In [36]:
final_row.shape

torch.Size([1, 2, 1536, 1024])